# Urban Heat & Cooling-Priority Mapping — Season Window Diagnostic (C4)

**NUS-ISS Practice Module, Week 2.** Tests candidate season definitions
against actual Landsat scene counts, so C4's "season-controlled" window gets
picked from real numbers instead of a guess. Motivated by: a Feb-Apr /
2021-2024 / cloud<20% guess produced only 2 usable Landsat scenes — this
notebook checks several physically-motivated alternatives before you lock
one in.

Candidates tested, all targeting peak-heat conditions (relevant for a tool
informing permanent landscape decisions, not just "the dry season"
generically):

- **A — Apr/May**: first inter-monsoon period (weak winds, low cloud, high insolation)
- **B — Oct/Nov**: second inter-monsoon period
- **C — Apr/May + Oct/Nov**: both inter-monsoon periods combined
- **D — Feb/Mar/Apr**: the earlier placeholder guess, kept for comparison
- **E — all 12 months**: reference upper bound (how many scenes exist at all, no season restriction)

Each is tested at both `CLOUD_COVER_MAX = 70` (Week-1's actual locked value)
and `= 20` (the earlier incorrect guess), to also show how much of the
2-scene problem was the season filter vs. the cloud threshold.

One notebook, run top to bottom:

1. **Setup** — install deps, authenticate Earth Engine
2. **SW.1** — config (AOI, years, candidate definitions)
3. **SW.2** — scene-count function
4. **SW.3** — run all candidates x both cloud thresholds
5. **SW.4** — results table + recommendation

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q earthengine-api pandas


## Setup 2 — Authenticate & initialize Earth Engine

In [2]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- your GCP project

if PROJECT_ID == "your-gcp-project-id":
    raise ValueError("PROJECT_ID is still the placeholder. Set it, then re-run this cell.")

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


---
# SW — Season Window Diagnostic


## SW.1 — Config

In [3]:
# --- SW CELL 1: Config -------------------------------------------------------
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

# Multi-year range for C4 robustness — wider than Week-1's rolling 2-year
# window since we specifically want MULTIPLE occurrences of the same season.
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
# Note: 2026 is partial (only through ~July at time of writing) — Apr/May 2026
# already happened, Oct/Nov 2026 has not, so candidate B/C will show fewer
# 2026 scenes than other years for that reason alone, not a data problem.

CANDIDATES = {
    "A: Apr-May (inter-monsoon 1)": [4, 5],
    "B: Oct-Nov (inter-monsoon 2)": [10, 11],
    "C: Apr/May + Oct/Nov (both inter-monsoon)": [4, 5, 10, 11],
    "D: Feb-Apr (earlier placeholder guess)": [2, 3, 4],
    "E: All 12 months (reference, no season restriction)": list(range(1, 13)),
}

CLOUD_THRESHOLDS = [70, 20]  # Week-1's actual value, then the earlier incorrect guess

print(f"Years tested: {YEARS}")
for name, months in CANDIDATES.items():
    print(f"  {name}: months {months}")


Years tested: [2021, 2022, 2023, 2024, 2025, 2026]
  A: Apr-May (inter-monsoon 1): months [4, 5]
  B: Oct-Nov (inter-monsoon 2): months [10, 11]
  C: Apr/May + Oct/Nov (both inter-monsoon): months [4, 5, 10, 11]
  D: Feb-Apr (earlier placeholder guess): months [2, 3, 4]
  E: All 12 months (reference, no season restriction): months [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


## SW.2 — Scene-count function

In [4]:
# --- SW CELL 2: Scene-count function -----------------------------------------
def date_filter_for_years_months(collection, years, months):
    filters = []
    for y in years:
        for m in months:
            start = ee.Date.fromYMD(y, m, 1)
            end = start.advance(1, "month")
            filters.append(ee.Filter.date(start, end))
    return collection.filter(ee.Filter.Or(*filters))


def count_landsat_scenes(months, cloud_max):
    l8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(sg_bbox)
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_max))
    )
    l9 = (
        ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        .filterBounds(sg_bbox)
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_max))
    )
    merged = l8.merge(l9)
    filtered = date_filter_for_years_months(merged, YEARS, months)
    return filtered.size().getInfo()

print("Scene-count function defined.")


Scene-count function defined.


## SW.3 — Run all candidates x both cloud thresholds

This makes real (non-cached) calls per candidate/threshold combo — 10 total
`.getInfo()` calls, each a fresh scene count. Takes a minute or two.


In [5]:
# --- SW CELL 3: Run all candidates --------------------------------------------
import pandas as pd

rows = []
for name, months in CANDIDATES.items():
    for cloud_max in CLOUD_THRESHOLDS:
        n = count_landsat_scenes(months, cloud_max)
        rows.append({"candidate": name, "cloud_cover_max": cloud_max, "n_scenes": n})
        print(f"  {name} | cloud<{cloud_max}: {n} scenes")

results_df = pd.DataFrame(rows)


  A: Apr-May (inter-monsoon 1) | cloud<70: 29 scenes
  A: Apr-May (inter-monsoon 1) | cloud<20: 6 scenes
  B: Oct-Nov (inter-monsoon 2) | cloud<70: 16 scenes
  B: Oct-Nov (inter-monsoon 2) | cloud<20: 1 scenes
  C: Apr/May + Oct/Nov (both inter-monsoon) | cloud<70: 45 scenes
  C: Apr/May + Oct/Nov (both inter-monsoon) | cloud<20: 7 scenes
  D: Feb-Apr (earlier placeholder guess) | cloud<70: 43 scenes
  D: Feb-Apr (earlier placeholder guess) | cloud<20: 4 scenes
  E: All 12 months (reference, no season restriction) | cloud<70: 140 scenes
  E: All 12 months (reference, no season restriction) | cloud<20: 17 scenes


## SW.4 — Results table + recommendation

No hard rule for "enough" scenes, but common guidance for a defensible
multi-year median composite is roughly 8-10+ clear scenes. Below that,
you're closer to averaging a handful of arbitrary dates than computing a
robust seasonal signal — worth treating as a stated limitation if you end
up there.


In [6]:
# --- SW CELL 4: Results table + recommendation --------------------------------
pivot = results_df.pivot(index="candidate", columns="cloud_cover_max", values="n_scenes")
pivot = pivot[[70, 20]]  # Week-1's value first, then the earlier guess, for direct comparison
print(pivot.to_string())

print("\n--- Recommendation guidance (not an automatic decision) ---")
MIN_DEFENSIBLE_SCENES = 8
for name in CANDIDATES:
    n70 = pivot.loc[name, 70]
    flag = "✅ likely defensible" if n70 >= MIN_DEFENSIBLE_SCENES else "⚠️  thin — treat as a stated limitation if used"
    print(f"  {name}: {n70} scenes at cloud<70 -> {flag}")

print("\nPick based on this table plus the physical justification (inter-monsoon =")
print("peak-heat conditions, most relevant for a tool informing permanent decisions).")
print("Whatever you pick, set YEARS/DRY_SEASON_MONTHS/CLOUD_COVER_MAX identically in")
print("BOTH gee_heat_variants.ipynb and adaptive_capacity_pillar.ipynb.")


cloud_cover_max                                       70  20
candidate                                                   
A: Apr-May (inter-monsoon 1)                          29   6
B: Oct-Nov (inter-monsoon 2)                          16   1
C: Apr/May + Oct/Nov (both inter-monsoon)             45   7
D: Feb-Apr (earlier placeholder guess)                43   4
E: All 12 months (reference, no season restriction)  140  17

--- Recommendation guidance (not an automatic decision) ---
  A: Apr-May (inter-monsoon 1): 29 scenes at cloud<70 -> ✅ likely defensible
  B: Oct-Nov (inter-monsoon 2): 16 scenes at cloud<70 -> ✅ likely defensible
  C: Apr/May + Oct/Nov (both inter-monsoon): 45 scenes at cloud<70 -> ✅ likely defensible
  D: Feb-Apr (earlier placeholder guess): 43 scenes at cloud<70 -> ✅ likely defensible
  E: All 12 months (reference, no season restriction): 140 scenes at cloud<70 -> ✅ likely defensible

Pick based on this table plus the physical justification (inter-monsoon =
peak-